# Experiment 4: Vision Benchmark Stress Test

**Paper**: Engineering Truthiness: A Standard for Pseudo–Ground Truth in Machine Learning Evaluation

**Description**: CIFAR-10 with asymmetric class-conditional noise. Demonstrates a high-SRI regime where structured proxy disagreement creates conditions for ranking instability.

**Requirements**: See `requirements.txt` in the repository root.

**Usage**: Run cells sequentially. All outputs are saved to `outputs/{exp_name}/`.

---

In [ ]:
# =========================
# CONFIGURATION
# =========================
# Public mode: all data is generated synthetically or loaded from public sources.
# To use private data, create config/config_paths_private.py (gitignored).

import os, sys
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Optional: mount Google Drive for private data
    # from google.colab import drive
    # drive.mount('/content/drive')
    pass

# Resolve repo root (works from notebooks/ or repo root)
_nb_dir = Path('.').resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir

# Try private config first, fall back to public defaults
try:
    sys.path.insert(0, str(_repo_root / 'config'))
    from config_paths_private import *  # noqa: F403
    CONFIG_MODE = 'private'
    print('[INFO] Using private path configuration')
except ImportError:
    # Public defaults — everything runs with synthetic data
    REPO_ROOT = _repo_root
    RUN_ID = 'public_run'
    DATADIR = REPO_ROOT / 'data'
    OUTDIR = REPO_ROOT / 'outputs'
    CONFIG_MODE = 'public'
    print('[INFO] Using public path configuration')

Path(OUTDIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# =========================
# SHARED UTILITIES
# =========================
import os, json, time, platform, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

def now_utc_compact():
    import datetime
    return datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S_utc")

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def bootstrap_ci(x, n=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    stats = []
    for _ in range(n):
        samp = rng.choice(x, size=len(x), replace=True)
        stats.append(np.mean(samp))
    lo = np.quantile(stats, alpha/2)
    hi = np.quantile(stats, 1 - alpha/2)
    return (float(np.mean(x)), float(lo), float(hi))

def make_run_dirs(exp_name: str):
    run_id = f"{exp_name}_{now_utc_compact()}"
    base = Path("outputs") / exp_name / run_id
    figs = base / "figures"
    tabs = base / "tables"
    logs = base / "logs"
    for d in [base, figs, tabs, logs]:
        safe_mkdir(d)
    return run_id, base, figs, tabs, logs

def write_manifest(base: Path, meta: dict):
    mf = base / "run_manifest.json"
    meta = dict(meta)
    meta["platform"] = {
        "python": platform.python_version(),
        "platform": platform.platform()
    }
    mf.write_text(json.dumps(meta, indent=2))
    return mf

set_global_seed(42)


In [ ]:
# =========================
# 04 — SETUP + CIFAR LOAD
# =========================
EXP = "04_vision"
run_id, out_base, out_figs, out_tabs, out_logs = make_run_dirs(EXP)

# Try to load real CIFAR-10; fall back to synthetic if torch unavailable
try:
    import torch
    from torchvision.datasets import CIFAR10
    from torchvision import transforms

    transform = transforms.Compose([transforms.ToTensor()])
    trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
    class_names = trainset.classes
    K = 10
    y_gold = np.array([trainset[i][1] for i in range(len(trainset))], dtype=int)
    USE_REAL_CIFAR = True
    print(f"[INFO] Loaded real CIFAR-10: {len(trainset)} samples, {K} classes")

except (ImportError, OSError) as e:
    # Synthetic fallback: simulate CIFAR-10 label distribution
    print(f"[INFO] torch/CIFAR unavailable ({e}), using synthetic labels")
    K = 10
    class_names = ["airplane", "automobile", "bird", "cat", "deer",
                    "dog", "frog", "horse", "ship", "truck"]
    rng_cifar = np.random.default_rng(42)
    # 50,000 samples with uniform class distribution (like CIFAR-10)
    y_gold = np.repeat(np.arange(K), 5000)
    rng_cifar.shuffle(y_gold)
    USE_REAL_CIFAR = False
    print(f"[INFO] Synthetic CIFAR-10 proxy: {len(y_gold)} samples, {K} classes")

len(y_gold), class_names


In [ ]:
# =========================
# 04 — SILVER LABELS + TRANSITION + FIDELITY
# =========================
rng = np.random.default_rng(42)
noise_p = 0.2  # keep consistent with paper claims; change if needed

# y_gold is set in the previous cell (from real CIFAR-10 or synthetic fallback)
# Only extract from trainset if we loaded real CIFAR
if "trainset" in dir():
    y_gold = np.array([trainset[i][1] for i in range(len(trainset))], dtype=int)
y_silver = y_gold.copy()

flip = rng.random(len(y_silver)) < noise_p
# random wrong class when flipped
y_silver[flip] = rng.integers(0, K, size=flip.sum())
# ensure changed label (optional strictness)
mask_same = (y_silver == y_gold) & flip
y_silver[mask_same] = (y_silver[mask_same] + 1) % K

# transition matrix T[g, s] = P(silver=s | gold=g)
T = np.zeros((K,K), dtype=float)
for g in range(K):
    idx = np.where(y_gold == g)[0]
    for s in range(K):
        T[g,s] = np.mean(y_silver[idx] == s) if len(idx) else 0.0

# per-class fidelity = P(silver==gold | gold=c) = diag(T)
fidelity = np.diag(T)

df04 = pd.DataFrame({
    "class_id": np.arange(K),
    "class_name": class_names,
    "fidelity": fidelity
})
df04


In [ ]:
# =========================
# 04 — SUBGROUP ANALYSIS (ANIMALS VS VEHICLES)
# =========================
animal = {"bird","cat","deer","dog","frog","horse"}
vehicle = {"airplane","automobile","ship","truck"}

df04["subgroup"] = df04["class_name"].apply(lambda x: "animal" if x in animal else "vehicle")
sub_stats = df04.groupby("subgroup")["fidelity"].agg(["mean","std","min","max"]).reset_index()

# simple t-test (lightweight)
from scipy import stats
a = df04[df04["subgroup"]=="animal"]["fidelity"].values
v = df04[df04["subgroup"]=="vehicle"]["fidelity"].values
t_stat, p_val = stats.ttest_ind(a, v, equal_var=False)

sub_stats, (t_stat, p_val)


In [ ]:
# =========================
# 04 — SAVE ARTIFACTS
# =========================
df04.to_csv(out_tabs / "results_04_vision_fidelity.csv", index=False)
sub_stats.to_csv(out_tabs / "results_04_vision_subgroup.csv", index=False)

# save full transition matrix
dfT = pd.DataFrame(T, columns=[f"silver_{i}" for i in range(K)])
dfT.insert(0, "gold_class", np.arange(K))
dfT.to_csv(out_tabs / "results_04_transition_matrix.csv", index=False)

import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.bar(df04["class_name"], df04["fidelity"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Label fidelity  P(silver=gold | gold=c)")
plt.title(f"CIFAR-10 proxy fidelity by class (noise_p={noise_p})")
plt.tight_layout()
plt.savefig(out_figs / "fig01_cifar_fidelity_by_class.png", dpi=300)

manifest = write_manifest(out_base, {
    "experiment": EXP,
    "run_id": run_id,
    "noise_p": noise_p,
    "t_test_animals_vs_vehicles": {"t": float(t_stat), "p": float(p_val)},
    "outputs": {
        "tables": [
            str(out_tabs / "results_04_vision_fidelity.csv"),
            str(out_tabs / "results_04_vision_subgroup.csv"),
            str(out_tabs / "results_04_transition_matrix.csv"),
        ],
        "figures": [str(out_figs / "fig01_cifar_fidelity_by_class.png")]
    }
})
manifest


---
## Appendix Figures: Experiment 4 (Vision Benchmark Stress Test)
The following figures are designed for direct inclusion in the paper appendix.

In [ ]:
# =========================
# APPENDIX FIG D1: Full Transition Matrix Heatmap
# =========================
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(T, cmap='Blues', vmin=0, vmax=1)

# Annotate every cell
for i in range(K):
    for j in range(K):
        val = T[i, j]
        color = 'white' if val > 0.4 else 'black'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(K))
ax.set_yticks(range(K))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Silver Label', fontsize=12)
ax.set_ylabel('Gold Label', fontsize=12)
ax.set_title('Figure D1. Full Noise Transition Matrix T[gold, silver] for CIFAR-10\n'
             f'P(silver=s | gold=g) under {noise_p:.0%} asymmetric class-conditional noise.\n'
             'Diagonal entries represent label fidelity. Off-diagonal entries show confusion patterns.',
             fontsize=11, pad=15)
plt.colorbar(im, ax=ax, label='P(silver | gold)', shrink=0.8)
plt.tight_layout()
plt.savefig(out_figs / 'figD1_transition_matrix_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SAVED] figD1_transition_matrix_heatmap.png')


In [ ]:
# =========================
# APPENDIX FIG D2: Fidelity by Subgroup (Animals vs Vehicles) with Error Bars
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-class fidelity colored by subgroup
ax = axes[0]
colors = ['#e74c3c' if s == 'animal' else '#3498db' for s in df04['subgroup']]
bars = ax.bar(df04['class_name'], df04['fidelity'], color=colors, alpha=0.8, edgecolor='white')
ax.axhline(df04['fidelity'].mean(), color='gray', linestyle='--', alpha=0.5,
           label=f'Overall mean = {df04["fidelity"].mean():.4f}')
ax.set_ylabel('Label Fidelity P(silver=gold | gold=c)', fontsize=11)
ax.set_title('Per-Class Fidelity', fontsize=12)
ax.set_xticklabels(df04['class_name'], rotation=45, ha='right')
ax.legend(fontsize=10)

# Add custom legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', alpha=0.8, label='Animal'),
                   Patch(facecolor='#3498db', alpha=0.8, label='Vehicle')]
ax.legend(handles=legend_elements, fontsize=10, loc='lower left')

# Subgroup comparison
ax = axes[1]
for sg in ['animal', 'vehicle']:
    vals = df04[df04['subgroup'] == sg]['fidelity']
    color = '#e74c3c' if sg == 'animal' else '#3498db'
    ax.boxplot(vals, positions=[0 if sg == 'animal' else 1], widths=0.4,
              patch_artist=True, boxprops=dict(facecolor=color, alpha=0.6),
              medianprops=dict(color='black', linewidth=2))

ax.set_xticks([0, 1])
ax.set_xticklabels(['Animals\n(6 classes)', 'Vehicles\n(4 classes)'])
ax.set_ylabel('Label Fidelity', fontsize=11)
ax.set_title(f'Subgroup Comparison (t={sub_stats["mean"].values[0]:.3f} vs {sub_stats["mean"].values[1]:.3f},\n'
             f'p={p_val:.3f})', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Figure D2. Label Fidelity by Class and Subgroup (CIFAR-10)\n'
             'Red = animal classes, Blue = vehicle classes.\n'
             f'Animals mean={sub_stats["mean"].values[0]:.4f}, Vehicles mean={sub_stats["mean"].values[1]:.4f}.',
             fontsize=11, y=1.06)
plt.tight_layout()
plt.savefig(out_figs / 'figD2_fidelity_subgroup_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SAVED] figD2_fidelity_subgroup_comparison.png')


In [ ]:
# =========================
# APPENDIX FIG D3: Off-Diagonal Confusion Pattern Analysis
# Which class pairs are most confused under proxy noise?
# =========================
fig, ax = plt.subplots(figsize=(10, 6))

# Extract top off-diagonal confusion pairs
confusions = []
for i in range(K):
    for j in range(K):
        if i != j:
            confusions.append({
                'gold': class_names[i],
                'silver': class_names[j],
                'prob': T[i, j]
            })

import pandas as pd
df_conf = pd.DataFrame(confusions).sort_values('prob', ascending=False).head(15)

labels = [f'{r["gold"]} -> {r["silver"]}' for _, r in df_conf.iterrows()]
vals = df_conf['prob'].values

# Color by whether it is within-subgroup or cross-subgroup confusion
animal_set = {'bird','cat','deer','dog','frog','horse'}
vehicle_set = {'airplane','automobile','ship','truck'}
bar_colors = []
for _, r in df_conf.iterrows():
    g_sub = 'animal' if r['gold'] in animal_set else 'vehicle'
    s_sub = 'animal' if r['silver'] in animal_set else 'vehicle'
    bar_colors.append('#e74c3c' if g_sub == s_sub else '#3498db')

ax.barh(range(len(labels)), vals, color=bar_colors, alpha=0.8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('P(silver=j | gold=i)', fontsize=12)
ax.set_title('Figure D3. Top 15 Class Confusion Pairs Under Proxy Noise\n'
             'Red = within-subgroup confusion, Blue = cross-subgroup confusion.\n'
             'Shows which gold-silver class pairs are most frequently confused.',
             fontsize=11, pad=15)
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', alpha=0.8, label='Within subgroup'),
                   Patch(facecolor='#3498db', alpha=0.8, label='Cross subgroup')]
ax.legend(handles=legend_elements, fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(out_figs / 'figD3_top_confusion_pairs.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SAVED] figD3_top_confusion_pairs.png')


In [ ]:
# =========================
# APPENDIX TABLE D1: Full Fidelity and Transition Summary
# =========================
print('Table D1. CIFAR-10 Per-Class Label Fidelity and Dominant Confusion Target')
print('=' * 85)
print(f'{"Class":>12s}  {"Subgroup":>9s}  {"Fidelity":>9s}  {"Top Confusion":>16s}  {"Conf. Prob":>10s}')
print('-' * 85)

for _, row in df04.iterrows():
    cid = int(row['class_id'])
    # Find top off-diagonal confusion target
    off_diag = T[cid].copy()
    off_diag[cid] = 0  # exclude self
    top_target = np.argmax(off_diag)
    top_prob = off_diag[top_target]
    print(f'{row["class_name"]:>12s}  {row["subgroup"]:>9s}  {row["fidelity"]:>9.4f}  '
          f'{class_names[top_target]:>16s}  {top_prob:>10.4f}')

print('=' * 85)
print(f'Noise level: {noise_p:.0%}')
print(f'Data source: {"Real CIFAR-10" if USE_REAL_CIFAR else "Synthetic"}')
print(f'Animal mean fidelity: {sub_stats["mean"].values[0]:.4f} (std={sub_stats["std"].values[0]:.4f})')
print(f'Vehicle mean fidelity: {sub_stats["mean"].values[1]:.4f} (std={sub_stats["std"].values[1]:.4f})')
print(f't-test: t={t_stat:.3f}, p={p_val:.3f}')


In [ ]:
from IPython.display import Image, display as ipy_display
import glob

print("Saved figures:")
for p in sorted(glob.glob(str(out_figs / "*.png"))):
    print(" -", p)
    ipy_display(Image(filename=p))


In [ ]:
# ==============================
# END OF NOTEBOOK SUMMARY
# ==============================
import json
summary = {
    "run_dir": str(out_base),
    "fig_dir": str(out_figs),
    "tables_dir": str(out_tabs),
}
summary_path = out_base / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("[DONE] Outputs saved under:", out_base)
